# 06 · Model choice

Feature set S (distance, equipment, weight, day of week, coordinates) with a few GBM settings, compared against a log-linear model.

## Compare models on the three folds and the unseen-city test

In [1]:
import sys; sys.path.append("../src")
import pandas as pd
from gbm import GBMModel
from linear import LogLinearModel
from cv import run_cv, run_hard

raw = pd.read_csv("../data/train_test.csv")
FEATS = ["base", "weight", "dow", "coords"]   # feature set S

candidates = {
    "GBM default":           lambda: GBMModel(FEATS),
    "GBM small leaves":      lambda: GBMModel(FEATS, max_leaf_nodes=15, min_samples_leaf=100),
    "GBM regularised":       lambda: GBMModel(FEATS, max_leaf_nodes=15, min_samples_leaf=200,
                                              learning_rate=0.03, max_iter=400, l2_regularization=1.0),
    "GBM larger leaves":     lambda: GBMModel(FEATS, max_leaf_nodes=63, min_samples_leaf=30),
    "LogLinear ridge a=1":   lambda: LogLinearModel(1.0),
    "LogLinear ridge a=10":  lambda: LogLinearModel(10.0),
}

rows = []
for name, make in candidates.items():
    cv, hard = run_cv(make, raw), run_hard(make, raw, seeds=(0, 1, 2))
    rows.append({"model": name,
                 "MAE_clean cv": cv["MAE_clean"].mean(),
                 "fold1": cv.loc[1, "MAE_clean"], "fold2": cv.loc[2, "MAE_clean"],
                 "fold3": cv.loc[3, "MAE_clean"],
                 "MAPE%_clean": cv["MAPE%_clean"].mean(),
                 "MAE (all rows)": cv["MAE"].mean(),
                 "hard subset": hard["MAE_clean"].mean()})

display(pd.DataFrame(rows).set_index("model").round(2))

,MAE_clean cv,fold1,fold2,fold3,MAPE%_clean,MAE (all rows),hard subset
model,,,,,,,
GBM default,78.86,142.14,51.99,42.46,3.23,132.42,48.65
GBM small leaves,78.76,141.90,51.91,42.47,3.22,132.33,47.93
GBM regularised,78.78,141.95,51.82,42.57,3.22,132.34,47.85
GBM larger leaves,79.04,142.09,52.47,42.56,3.23,132.60,49.24
LogLinear ridge a=1,89.91,145.01,65.52,59.21,3.84,143.38,64.55
LogLinear ridge a=10,89.97,145.24,65.50,59.16,3.84,143.43,64.41


## Choice

All the GBM settings are within about $0.30 of each other, so the exact settings barely matter. I use **"GBM small leaves"** (15 leaves, at least 100 rows per leaf): it ties for best, and bigger leaves are a little safer on a period the model hasn't seen. The log-linear model is about $10–17 worse per fold, so the non-linear model is worth it.

The GBM is trained on log(rate per mile) and its prediction is multiplied back by distance. Fold 1 stays weak for every model: May–Jun was a market peak that nothing in the training months anticipated.